# OLS Combination: Granger-Ramanathan Approach

**Granger & Ramanathan (1984)** proposed combining forecasts via **OLS regression**,
where the actual values are regressed on the individual forecasts:

$$y_t = \beta_0 + \sum_{k=1}^{K} \beta_k \hat{y}_{k,t} + \varepsilon_t$$

They identified three variants:

1. **Variant 1**: No intercept, weights constrained to sum to 1 ($\beta_0 = 0$, $\sum \beta_k = 1$, $\beta_k \geq 0$)
2. **Variant 2**: With intercept, unconstrained ($\beta_0 \neq 0$, no restrictions on $\beta_k$)
3. **Variant 3**: No intercept, unconstrained ($\beta_0 = 0$, no restrictions on $\beta_k$)

**Topics covered:**
- All three Granger-Ramanathan variants
- Interpretation of weights and intercept
- Rolling OLS combination for out-of-sample evaluation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from forecastbox.combination import OLSCombiner, SimpleCombiner
from forecastbox.core.forecast import Forecast
from forecastbox.metrics import mae, rmse

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 5)
np.random.seed(42)

## 1. Granger-Ramanathan Framework

The key insight of Granger-Ramanathan is to treat forecast combination as a
**regression problem**. The actual value $y_t$ is the dependent variable, and
the individual forecasts $\hat{y}_{k,t}$ are the regressors:

$$y_t = \alpha + \sum_{k=1}^{K} \beta_k \hat{y}_{k,t} + \varepsilon_t$$

The estimated coefficients $\hat{\beta}_k$ serve as combination weights.

**Advantages over simple weighting:**
- Can correct for **bias** in individual forecasts (via the intercept)
- Can assign **negative weights** to models that are negatively correlated with the target
- Minimizes in-sample MSE of the combined forecast

In [ ]:
# Load data and split into train/test
df = pd.read_csv("../data/inflation_forecasts.csv", parse_dates=["date"])

n_train = 80
train_df = df.iloc[:n_train]
test_df = df.iloc[n_train:]

actual_train = train_df["actual"].values
actual_test = test_df["actual"].values

model_cols = ["fc_arima", "fc_ets", "fc_var", "fc_naive", "fc_drift"]
model_names = [c.replace("fc_", "").upper() for c in model_cols]

# Training arrays
forecasts_train = [train_df[col].values for col in model_cols]

# Test Forecast objects
forecasts_test = [
    Forecast(point=test_df[col].values, model_name=name)
    for col, name in zip(model_cols, model_names)
]

print(f"Training: {n_train} observations")
print(f"Test: {len(test_df)} observations")
print(f"Models: {model_names}")

## 2. Variant 1: No Intercept, Weights Sum to 1

This is the **constrained** version:

$$\min_{\boldsymbol{w}} \|\mathbf{y} - \mathbf{F}\boldsymbol{w}\|^2 \quad \text{s.t.} \quad \sum_{k=1}^{K} w_k = 1, \; w_k \geq 0$$

This ensures the weights are interpretable as proportions and the combination
is unbiased when individual forecasts are unbiased.

In [ ]:
# Variant 1: constrained, no intercept
gr1 = OLSCombiner(intercept=False, constrained=True)
gr1.fit(forecasts_train, actual_train)
fc_gr1 = gr1.combine(forecasts_test)

print("Variant 1: No Intercept, Weights Sum to 1")
print(f"  Intercept: {gr1.intercept_:.4f}")
print(f"  Weights (sum={np.sum(gr1.weights_):.4f}):")
for name, w in zip(model_names, gr1.weights_):
    print(f"    {name:8s}: {w:.4f}")
print(f"  MAE:  {mae(actual_test, fc_gr1.point):.4f}")
print(f"  RMSE: {rmse(actual_test, fc_gr1.point):.4f}")

## 3. Variant 2: With Intercept (bias correction)

Adding an intercept allows the combination to correct for **systematic bias**
in the individual forecasts:

$$y_t = \alpha + \sum_{k=1}^{K} \beta_k \hat{y}_{k,t} + \varepsilon_t$$

A **non-zero intercept** indicates that the individual forecasts have a
systematic bias that the combination can correct.

Weights are **unrestricted**: they can be negative and need not sum to 1.

In [ ]:
# Variant 2: with intercept, unconstrained
gr2 = OLSCombiner(intercept=True, constrained=False)
gr2.fit(forecasts_train, actual_train)
fc_gr2 = gr2.combine(forecasts_test)

print("Variant 2: With Intercept, Unconstrained")
print(f"  Intercept: {gr2.intercept_:.4f}")
print(f"  Weights (sum={np.sum(gr2.weights_):.4f}):")
for name, w in zip(model_names, gr2.weights_):
    print(f"    {name:8s}: {w:.4f}")
print(f"  MAE:  {mae(actual_test, fc_gr2.point):.4f}")
print(f"  RMSE: {rmse(actual_test, fc_gr2.point):.4f}")

# Interpret intercept
if abs(gr2.intercept_) > 0.01:
    direction = "upward" if gr2.intercept_ > 0 else "downward"
    print(f"\n  Interpretation: The intercept ({gr2.intercept_:.4f}) suggests a {direction}"
          f" bias correction is needed.")
else:
    print(f"\n  Interpretation: The intercept is near zero, individual forecasts are approximately unbiased.")

## 4. Variant 3: No Intercept, Unrestricted

This variant drops both the intercept and the sum-to-one constraint:

$$y_t = \sum_{k=1}^{K} \beta_k \hat{y}_{k,t} + \varepsilon_t$$

Weights can be **negative** (allowing models to offset each other) and
**do not sum to 1** (allowing the combination to scale).

This is the most flexible variant but also the most prone to overfitting.

In [ ]:
# Variant 3: no intercept, unconstrained
gr3 = OLSCombiner(intercept=False, constrained=False)
gr3.fit(forecasts_train, actual_train)
fc_gr3 = gr3.combine(forecasts_test)

print("Variant 3: No Intercept, Unrestricted")
print(f"  Intercept: {gr3.intercept_:.4f}")
print(f"  Weights (sum={np.sum(gr3.weights_):.4f}):")
for name, w in zip(model_names, gr3.weights_):
    sign = "+" if w >= 0 else ""
    print(f"    {name:8s}: {sign}{w:.4f}")
print(f"  MAE:  {mae(actual_test, fc_gr3.point):.4f}")
print(f"  RMSE: {rmse(actual_test, fc_gr3.point):.4f}")

# Compare all three variants visually
fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)
variants = [("Variant 1\n(Constrained)", gr1), ("Variant 2\n(With Intercept)", gr2), ("Variant 3\n(Unrestricted)", gr3)]
for ax, (title, gr) in zip(axes, variants):
    colors = ["steelblue" if w >= 0 else "salmon" for w in gr.weights_]
    ax.bar(model_names, gr.weights_, color=colors)
    ax.set_title(title)
    ax.axhline(y=0, color="black", linewidth=0.5)
    ax.grid(True, alpha=0.3, axis="y")
    ax.tick_params(axis="x", rotation=45)
axes[0].set_ylabel("Weight")
plt.suptitle("Granger-Ramanathan Weights by Variant", y=1.02)
plt.tight_layout()
plt.show()

## 5. Out-of-Sample Evaluation

A critical issue with OLS combination is **overfitting**: weights estimated on the
full training set may not generalize well. A more robust approach is **rolling OLS**:

1. At each test period $t$, use only observations $1, \ldots, t-1$ to estimate weights.
2. Apply these weights to produce the forecast for period $t$.
3. Move to the next period and repeat.

This simulates a real-time forecasting environment.

In [ ]:
# Rolling OLS combination
n_total = len(df)
window = 60  # minimum training window
rolling_predictions = {"GR1": [], "GR2": [], "GR3": [], "Simple Avg": []}
rolling_dates = []
rolling_actual = []

for t in range(window, n_total):
    # Training data: observations 0..t-1
    train_slice = df.iloc[:t]
    fc_train_t = [train_slice[col].values for col in model_cols]
    actual_t = train_slice["actual"].values

    # Test point: observation t (single horizon)
    test_point = [
        Forecast(point=np.array([df[col].iloc[t]]), model_name=name)
        for col, name in zip(model_cols, model_names)
    ]

    # GR Variant 1
    combiner1 = OLSCombiner(intercept=False, constrained=True)
    combiner1.fit(fc_train_t, actual_t)
    rolling_predictions["GR1"].append(combiner1.combine(test_point).point[0])

    # GR Variant 2
    combiner2 = OLSCombiner(intercept=True, constrained=False)
    combiner2.fit(fc_train_t, actual_t)
    rolling_predictions["GR2"].append(combiner2.combine(test_point).point[0])

    # GR Variant 3
    combiner3 = OLSCombiner(intercept=False, constrained=False)
    combiner3.fit(fc_train_t, actual_t)
    rolling_predictions["GR3"].append(combiner3.combine(test_point).point[0])

    # Simple Average (benchmark)
    simple_comb = SimpleCombiner(method="mean")
    simple_comb.fit(fc_train_t, actual_t)
    rolling_predictions["Simple Avg"].append(simple_comb.combine(test_point).point[0])

    rolling_dates.append(df["date"].iloc[t])
    rolling_actual.append(df["actual"].iloc[t])

rolling_actual = np.array(rolling_actual)

# Rolling evaluation
print("Rolling OLS Combination Results")
print("=" * 45)
for name, preds in rolling_predictions.items():
    preds_arr = np.array(preds)
    print(f"  {name:12s}  MAE={mae(rolling_actual, preds_arr):.4f}  "
          f"RMSE={rmse(rolling_actual, preds_arr):.4f}")

# Plot rolling predictions
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(rolling_dates, rolling_actual, "k-", linewidth=2, label="Actual")
for name, preds in rolling_predictions.items():
    ax.plot(rolling_dates, preds, "--", alpha=0.7, label=name)
ax.set_title("Rolling OLS Combination vs Actual")
ax.set_xlabel("Date")
ax.set_ylabel("Inflation")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Exercise 1: Which GR variant performs best on M4 sample?

Load `m4_sample.csv`, apply all 3 Granger-Ramanathan variants to each series,
and determine which variant produces the lowest average RMSE.

In [ ]:
# TODO: Exercise 1
# 1. Load m4_sample.csv
# 2. For each series_id, split into train/test (80/20)
# 3. Apply OLSCombiner with all 3 variant configurations
# 4. Compute RMSE for each variant on each series
# 5. Report average RMSE across series for each variant

## Exercise 2: Implement leave-one-out CV for weight estimation

Instead of a single train/test split, implement leave-one-out cross-validation
to estimate the OLS combination weights. Compare the CV-estimated weights
with the full-sample weights.

In [ ]:
# TODO: Exercise 2
# 1. For each observation t in the training set:
#    a. Remove observation t
#    b. Fit OLSCombiner on remaining observations
#    c. Predict observation t
# 2. Compute CV-based RMSE
# 3. Compare weights from full-sample vs average CV weights
# 4. Which approach gives better out-of-sample performance?